# 教程: 数据预处理 - 空间插值

## 目的
在水文模拟中，我们常常需要在没有直接观测数据的位置估算气象数据（如降雨）。例如，一个子流域的中心可能没有雨量站。在这种情况下，我们需要利用周边雨量站的数据，通过**空间插值（Spatial Interpolation）**来估算该位置的降雨。

本教程将展示一种常用且简单的空间插值方法——**反距离权重法（Inverse Distance Weighting, IDW）**。

此笔记本将：
1.  加载一个包含多个雨量站位置及其降雨记录的CSV文件。
2.  定义一个我们希望估算降雨的目标点（例如，子流域的形心）。
3.  实现反距离权重（IDW）算法。
4.  应用该算法，为目标点生成一个新的降雨时间序列。
5.  通过可视化地图和时间序列图，来展示插值过程和结果。

## 1. 环境设置

我们只需要`pandas`用于数据处理，`numpy`用于计算，以及`matplotlib`用于绘图。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. 加载数据和定义目标

我们加载包含三个雨量站（A, B, C）数据的文件，并定义我们感兴趣的目标点的位置。

In [ ]:
# 加载数据
df = pd.read_csv('../data/multi_gauge_rainfall.csv', index_col='timestamp', parse_dates=True)

# 定义站点信息
gauges = {
    'A': {'x': df['gauge_A_x'][0], 'y': df['gauge_A_y'][0], 'mm': df['gauge_A_mm']},
    'B': {'x': df['gauge_B_x'][0], 'y': df['gauge_B_y'][0], 'mm': df['gauge_B_mm']},
    'C': {'x': df['gauge_C_x'][0], 'y': df['gauge_C_y'][0], 'mm': df['gauge_C_mm']}
}

# 定义目标点
target_point = {'x': 45, 'y': 40}

print("站点和目标点定义完成。")
display(df)

### 可视化站点布局

In [ ]:
plt.figure(figsize=(8, 6))
for name, data in gauges.items():
    plt.plot(data['x'], data['y'], 'bo', markersize=10, label=f'Gauge {name}')
    plt.text(data['x']+2, data['y']+2, f'Gauge {name}')

plt.plot(target_point['x'], target_point['y'], 'r*', markersize=15, label='Target Point')
plt.text(target_point['x']+2, target_point['y']+2, 'Target')

plt.title('Spatial Layout of Rain Gauges and Target Point')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

## 3. 实现IDW插值算法

IDW的核心思想是，一个未知点的值是其周围已知点值的加权平均，而权重与距离成反比。距离越近的点，权重越大。公式为：

$$ V_{target} = \frac{\sum_{i=1}^{N} (V_i / d_i^p)}{\sum_{i=1}^{N} (1 / d_i^p)} $$

其中 `V` 是值（这里是降雨量），`d` 是距离，`p` 是一个幂指数（通常取2）。

我们编写一个函数，它接受一行数据（代表一个时间步），并计算出目标点的插值。

In [ ]:
def idw_interpolate(row, gauges_info, target_info, power=2):
    numerator = 0
    denominator = 0
    
    for name, info in gauges_info.items():
        # 计算距离
        distance = np.sqrt((info['x'] - target_info['x'])**2 + (info['y'] - target_info['y'])**2)
        
        # 避免除以零
        if distance < 1e-6:
            return row[f'gauge_{name}_mm']
            
        # 计算权重
        weight = 1.0 / (distance ** power)
        
        # 累加分子和分母
        numerator += row[f'gauge_{name}_mm'] * weight
        denominator += weight
        
    if denominator == 0:
        return np.nan
        
    return numerator / denominator

# 将函数应用到DataFrame的每一行
df['target_mm'] = df.apply(idw_interpolate, axis=1, args=(gauges, target_point))

print("插值计算完成:")
display(df[['gauge_A_mm', 'gauge_B_mm', 'gauge_C_mm', 'target_mm']])

## 4. 可视化插值结果

最后，我们绘制出所有雨量站的降雨过程线和我们新计算出的目标点的过程线，以进行比较。

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df.index, df['gauge_A_mm'], 'b:o', label='Gauge A')
plt.plot(df.index, df['gauge_B_mm'], 'g:s', label='Gauge B')
plt.plot(df.index, df['gauge_C_mm'], 'c:^', label='Gauge C')
plt.plot(df.index, df['target_mm'], 'r-o', linewidth=2, markersize=8, label='Interpolated Target')

plt.title('Spatially Interpolated Rainfall Time Series')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm)')
plt.legend()
plt.grid(True)
plt.show()

print("从图中可以看到，目标点的插值结果（红线）是其他几个站点值的加权平均，其形态介于其他几个站点之间，符合预期。")